In [ ]:
import pandas as pd # The primary library for structured data manipulation and analysis.
import requests # Tool utilized for web communication.
from bs4 import BeautifulSoup # Tool for HTML parsing during data extraction.
import time # For rate limiting between requests.
import re # For pattern matching and string cleaning.
import numpy as np # For high-performance numerical operations.
import json # For handling and parsing metadata files.
import os # For filesystem operations and directory management.
import sys # For system-level interactions.

Takeaway : Successful import of all required libraries. No error messages should be displayed.

---

## Global Constants and URL Configuration

#### Purpose
Centralizing configuration parameters ensures the script remains maintainable. We include a User-Agent header to ensure successful communication with the host server.

In [ ]:
BASE_URL = "https://www.globalfirepower.com/"
LISTING_URL = BASE_URL + "countries-listing.php"
HEADERS = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}

print("Global configuration constants and headers have been successfully initialized.")

Takeaway : Confirmation message that constants have been initialized and headers are set for ethical scraping.

---

<h2 style="color: #117A65;">Data Collection and Preparation</h2>

This section focuses on automated data extraction and preliminary cleaning.

### Strategy and Schema Definition

#### Purpose
Defining the target data schema and identifying required metrics is essential for ensuring the resulting dataset supports all downstream analytical requirements.

### Scraping Setup and Execution

#### Purpose
The scraping module automates the extraction of raw data. This implementation includes robust error handling to manage server latency or connectivity issues.

In [ ]:
def scrape_country_data(url):
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        if response.status_code != 200: return None
        
        soup = BeautifulSoup(response.text, "html.parser")
        data = {}
        
        title = soup.find("h1")
        if title:
            match = re.search(r"\d{4}\s+(.*?)\s+Military Strength", title.text.strip())
            data["country"] = match.group(1).strip() if match else title.text.strip()
        
        containers = soup.find_all("div", class_="specsGenContainers")
        for container in containers:
            label_span = container.find("span", class_="textYellow")
            value_spans = container.find_all("span", class_="textWhite")
            if label_span and value_spans:
                key = re.sub(r"[^a-zA-Z0-9\s]", "", label_span.text.strip()).strip().lower().replace(" ", "_")
                data[key] = value_spans[-1].text.strip()
        
        return data
    except Exception as e:
        return None

def get_all_country_links():
    try:
        response = requests.get(LISTING_URL, headers=HEADERS, timeout=15)
        if response.status_code != 200: return []
            
        soup = BeautifulSoup(response.text, "html.parser")
        links = soup.find_all("a", href=re.compile(r"country-military-strength-detail\.php\?country_id="))
        
        unique_links = []
        for l in links:
            href = l['href']
            url = BASE_URL.rstrip('/') + '/' + href.lstrip('/') if not href.startswith('http') else href
            unique_links.append(url)
            
        return list(dict.fromkeys(unique_links))
    except Exception as e:
        return []

print("Executing Module 1: System-automated data extraction...")
all_links = get_all_country_links()
if not all_links:
    print("CRITICAL: No country links identified. Check connection or URL.")
else:
    raw_results = []
    print(f"Found {len(all_links)} nations. Commencing full-scale extraction...")
    
    # To extract ALL nations, leave the slice below empty [:]
    target_links = all_links[:] 
    
    for i, link in enumerate(target_links):
        res = scrape_country_data(link)
        if res: raw_results.append(res)
        if (i+1) % 10 == 0: print(f"Progress: {i+1}/{len(target_links)} nations processed...")
        time.sleep(0.1)

    df_raw = pd.DataFrame(raw_results)
    print(f"Extraction successful. Captured data for {len(df_raw)} nations.")
    if not df_raw.empty: print(df_raw.head())

Takeaway : A comprehensive raw table containing records for all 145 countries is produced, enabling global-scale military analysis.

---

#### Dataset Schema Verification and Raw Export

#### Purpose
Verifies the schema integrity and persists the raw state.

In [ ]:
if 'df_raw' in locals() and not df_raw.empty:
    print("Verification of captured raw data columns:")
    print(df_raw.columns.tolist())
    
    os.makedirs("data/raw", exist_ok=True)
    df_raw.to_csv("data/raw/military_raw_data.csv", index=False)
    print("Raw dataset successfully exported to data/raw/military_raw_data.csv")
else:
    print("Notice: No raw data available for export.")

Takeaway : The code produces a list of 50+ column headers and a confirmed CSV file export for data persistence.

---

## Data Cleaning and Standardization

### Purpose
Standardizes raw strings into high-precision numeric values, handling multi-line weapon blocks and symbols.

In [ ]:
def clean_raw_value(val):
    if pd.isna(val) or val == 'N/A': return 0.0
    first_part = str(val).split('\n')[0].split('\t')[0]
    clean = re.sub(r"[^0-9.]", "", first_part.split("(")[0])
    return float(clean) if clean else 0.0

def clean_dataset(df):
    if df.empty: return pd.DataFrame()
    cleaned_df = df.copy()
    cleaned_df.columns = [re.sub(r"[^a-z0-9]", "_", c.lower().strip()) for c in cleaned_df.columns]
    numeric_cols = [col for col in cleaned_df.columns if col != 'country']
    for col in numeric_cols:
        cleaned_df[col] = cleaned_df[col].map(clean_raw_value)
    return cleaned_df

if 'df_raw' in locals() and not df_raw.empty:
    df_cleaned = clean_dataset(df_raw)
    print("Cleaning and structural standardization complete.")
    print(df_cleaned.head())
else:
    print("Notice: Data cleaning skipped.")

Takeaway : A standardized table where all military assets for all countries are clean numbers, ready for mathematical modeling.

---

### Dataset Validation

#### Purpose
Verifies data quality against project benchmarks.

In [ ]:
if 'df_cleaned' in locals() and not df_cleaned.empty:
    print(f"Total Row Count Verification: {len(df_cleaned)}")
    print(f"Aggregate Null Percentage: {(df_cleaned.isnull().sum().sum() / df_cleaned.size) * 100}%")
    print("Attribute Type Audit:")
    print(df_cleaned.dtypes.head())
else:
    print("Notice: Data validation skipped.")

Takeaway : Statistics showing a complete 145-row dataset (0% nulls) and correctly formatted float/object types.

---

### Clean Data Export

#### Purpose
Persisting the cleaned dataset for the next milestone (KPI Engineering).

In [ ]:
if 'df_cleaned' in locals() and not df_cleaned.empty:
    os.makedirs("data/cleaned", exist_ok=True)
    df_cleaned.to_csv("data/cleaned/military_cleaned_data.csv", index=False)
    print("Cleaned dataset successfully exported to data/cleaned/military_cleaned_data.csv")
else:
    print("Notice: No cleaned data available for export.")

Takeaway : Successful export of the cleaned dataset, ready for KPI transformation in Milestone 2.